Experiment 4 -- Robustness to proxy-correlation strength and base-rate gap

In [1]:
# 04_sensitivity_analysis.ipynb
#
# Experiment 4 -- Robustness to proxy-correlation strength and base-rate gap
#
# Purpose:
#   Test whether the governance engine's behavior is an artifact of one
#   parameter choice by sweeping (a) proxy-correlation strength and
#   (b) between-group base-rate gap, reporting baseline vs gov_engine DIR and
#   gov_engine AUC across the grid (10 seeds each, 95% CI).
#
# Outputs:
#   results/tables/exp4_sensitivity.csv
#   results/figures/exp4_sensitivity_proxy.(png|pdf)

# ==== Imports and grayscale academic style (600 dpi, PNG+PDF, no captions) ====
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({
    "figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85",
})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def disparate_impact_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)
    r1 = approve[group == 1].mean(); r0 = approve[group == 0].mean()
    return min(r1, r0) / max(r1, r0) if max(r1, r0) > 0 else np.nan

def equalized_odds_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx = (y_true == ct) & mask
        return (y_pred[idx] == 1).mean() if idx.sum() else np.nan
    a, b = group == 1, group == 0
    return abs(rate(1, a) - rate(1, b)), abs(rate(0, a) - rate(0, b))



from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

FEATS = ["residential_region", "spending_pattern", "income", "debt_ratio", "pay_history"]

def synth(proxy_corr, base_rate, seed):
    rng = np.random.default_rng(seed); N = 8000
    protected = rng.binomial(1, 0.35, N)
    rr = proxy_corr * protected + rng.normal(0, 0.6, N)
    sp = proxy_corr * 0.9 * protected + rng.normal(0, 0.6, N)
    inc = rng.normal(0, 1, N); dr = rng.normal(0, 1, N); ph = rng.normal(0, 1, N)
    logit = -0.9*inc + 0.8*dr - 0.7*ph + 1.3*rr + 1.1*sp + rng.normal(0, 0.5, N)
    default = (logit > np.quantile(logit, 1 - base_rate)).astype(int)
    X = pd.DataFrame(dict(residential_region=rr, spending_pattern=sp,
                          income=inc, debt_ratio=dr, pay_history=ph))
    return X, default, protected

def evaluate(proxy_corr, base_rate, seed):
    X, y, g = synth(proxy_corr, base_rate, seed)
    Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, g, test_size=0.3, random_state=seed)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    p = m.predict_proba(Xte.values)[:, 1]; auc = roc_auc_score(yte, p)
    gg = gte
    yb = (p >= 0.5).astype(int); base = (yb == 0).mean()
    t1 = np.quantile(p[gg == 1], base); t0 = np.quantile(p[gg == 0], base)
    yg = np.where(gg == 1, (p >= t1), (p >= t0)).astype(int)
    return disparate_impact_ratio(yb, gg), disparate_impact_ratio(yg, gg), auc

SEEDS = list(range(10))
rows = []
# (a) sweep proxy correlation at fixed base rate 0.30
for pc in [0.4, 0.8, 1.2, 1.6, 2.0]:
    arr = np.array([evaluate(pc, 0.30, s) for s in SEEDS])
    m = arr.mean(0); ci = 1.96 * arr.std(0) / np.sqrt(len(SEEDS))
    rows.append({"sweep": "proxy_corr", "value": pc,
                 "baseline_DIR": m[0], "gov_DIR": m[1], "gov_AUC": m[2],
                 "baseline_DIR_ci": ci[0], "gov_DIR_ci": ci[1], "gov_AUC_ci": ci[2]})
# (b) sweep base-rate at fixed proxy correlation 1.6
for br in [0.15, 0.25, 0.35, 0.45]:
    arr = np.array([evaluate(1.6, br, s) for s in SEEDS])
    m = arr.mean(0); ci = 1.96 * arr.std(0) / np.sqrt(len(SEEDS))
    rows.append({"sweep": "base_rate", "value": br,
                 "baseline_DIR": m[0], "gov_DIR": m[1], "gov_AUC": m[2],
                 "baseline_DIR_ci": ci[0], "gov_DIR_ci": ci[1], "gov_AUC_ci": ci[2]})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR, "exp4_sensitivity.csv"), index=False)
print(res.round(3).to_string(index=False))

pc_df = res[res["sweep"] == "proxy_corr"]
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.errorbar(pc_df["value"], pc_df["baseline_DIR"], yerr=pc_df["baseline_DIR_ci"],
            marker="s", color="#999999", linewidth=1.5, capsize=3, label="Baseline DIR")
ax.errorbar(pc_df["value"], pc_df["gov_DIR"], yerr=pc_df["gov_DIR_ci"],
            marker="o", color="#000000", linewidth=1.5, capsize=3, label="AI-Gov-Alt DIR")
ax.errorbar(pc_df["value"], pc_df["gov_AUC"], yerr=pc_df["gov_AUC_ci"],
            marker="^", color="#555555", linewidth=1.5, linestyle="--", capsize=3, label="AI-Gov-Alt AUC")
ax.axhline(0.8, color="black", linestyle=":", linewidth=0.9)
ax.set_xlabel("Proxy-correlation strength"); ax.set_ylabel("Metric value")
ax.set_ylim(0, 1.05); ax.legend(frameon=True, edgecolor="black", loc="center right")
fig.tight_layout(); save_fig(fig, "exp4_sensitivity_proxy"); plt.close(fig)
print("Saved sensitivity table and figure.")


     sweep  value  baseline_DIR  gov_DIR  gov_AUC  baseline_DIR_ci  gov_DIR_ci  gov_AUC_ci
proxy_corr   0.40         0.764    1.000    0.979            0.007         0.0       0.001
proxy_corr   0.80         0.565    1.000    0.982            0.008         0.0       0.001
proxy_corr   1.20         0.404    1.000    0.987            0.011         0.0       0.001
proxy_corr   1.60         0.291    1.000    0.992            0.010         0.0       0.000
proxy_corr   2.00         0.211    1.000    0.995            0.009         0.0       0.001
 base_rate   0.15         0.605    1.000    0.993            0.008         0.0       0.001
 base_rate   0.25         0.376    1.000    0.992            0.011         0.0       0.001
 base_rate   0.35         0.220    0.999    0.991            0.009         0.0       0.001
 base_rate   0.45         0.127    0.999    0.991            0.009         0.0       0.001


Saved sensitivity table and figure.
